# Session 5: Database Commands

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-05_admin_helpers.py`

In [2]:
# Session 5 · Administrative work through driver helpers
import bson
from store_db import get_db

db = get_db()
client = db.client

print("ping           ->", client.admin.command("ping"))
print("databases      ->", client.list_database_names())
print("collections    ->", sorted(db.list_collection_names()))

db.create_collection("audit_log")
db.audit_log.insert_one({"event": "price_change", "productId": 101})
print("after create   ->", sorted(db.list_collection_names()))

db.audit_log.rename("price_audit")                  # renameCollection
print("after rename   ->", sorted(db.list_collection_names()))

db.drop_collection("price_audit")                   # drop
print("after drop     ->", sorted(db.list_collection_names()))

# A home-made version of dbStats / collStats using BSON sizes
def coll_stats(name):
    docs = list(db[name].find())
    sizes = [len(bson.encode(d)) for d in docs]
    return {"ns": f"{db.name}.{name}", "count": len(docs), "size": sum(sizes),
            "avgObjSize": round(sum(sizes) / len(sizes)), "nindexes": len(db[name].index_information())}

print("\n{:<14} {:>5} {:>7} {:>10} {:>8}".format("collection", "count", "bytes", "avgObjSize", "indexes"))
total = 0
for name in sorted(db.list_collection_names()):
    s = coll_stats(name); total += s["size"]
    print("{:<14} {:>5} {:>7} {:>10} {:>8}".format(name, s["count"], s["size"], s["avgObjSize"], s["nindexes"]))
print("dbStats-style total data size:", total, "bytes")

ping           -> {'ok': 1.0}
databases      -> ['kgs']
collections    -> ['customers', 'orders', 'products']
after create   -> ['audit_log', 'customers', 'orders', 'products']
after rename   -> ['customers', 'orders', 'price_audit', 'products']
after drop     -> ['customers', 'orders', 'products']

collection     count   bytes avgObjSize  indexes
customers          6     655        109        1
orders            10    1645        164        1
products          10    1742        174        1
dbStats-style total data size: 4042 bytes


## Example: `session-05_run_command.py`

In [3]:
# Session 5 · What a database command looks like
# Every helper (count, distinct, create, drop…) is sent to the server as a COMMAND DOCUMENT:
# the first key is the command name, its value is usually the collection.
# mongomock implements only a few raw commands, so this file includes a tiny dispatcher
# that maps command documents to driver calls. On a real server you would simply call
#     db.command(cmd)            (PyMongo)      or      db.runCommand(cmd)   (mongosh)
from store_db import get_db

db = get_db()

def run_command(cmd):
    name, target = next(iter(cmd.items()))
    if name == "count":
        return {"n": db[target].count_documents(cmd.get("query", {})), "ok": 1}
    if name == "distinct":
        return {"values": sorted(db[target].distinct(cmd["key"], cmd.get("query", {}))), "ok": 1}
    if name == "create":
        db.create_collection(target); return {"ok": 1}
    if name == "drop":
        db.drop_collection(target); return {"ns": f"{db.name}.{target}", "ok": 1}
    if name == "listCollections":
        return {"cursor": {"firstBatch": [{"name": n} for n in sorted(db.list_collection_names())]}, "ok": 1}
    if name == "createIndexes":
        made = [db[target].create_index(list(ix["key"].items()), name=ix["name"]) for ix in cmd["indexes"]]
        return {"createdIndexes": made, "ok": 1}
    if name == "find":
        docs = list(db[target].find(cmd.get("filter", {}), cmd.get("projection")).limit(cmd.get("limit", 0)))
        return {"cursor": {"firstBatch": docs, "ns": f"{db.name}.{target}"}, "ok": 1}
    return {"ok": 0, "errmsg": f"no such command: '{name}'", "code": 59}

commands = [
    {"count": "products", "query": {"category": "Accessories"}},
    {"distinct": "customers", "key": "city", "query": {"loyalty": {"$ne": "Bronze"}}},
    {"create": "wishlists"},
    {"listCollections": 1},
    {"createIndexes": "products", "indexes": [{"key": {"category": 1, "price": -1}, "name": "cat_price"}]},
    {"find": "products", "filter": {"stock": 0}, "projection": {"name": 1}, "limit": 1},
    {"drop": "wishlists"},
    {"fsyncUnlockEverything": 1},
]
for c in commands:
    print("db.runCommand(", c, ")")
    print("   ->", run_command(c), "\n")

db.runCommand( {'count': 'products', 'query': {'category': 'Accessories'}} )
   -> {'n': 4, 'ok': 1} 

db.runCommand( {'distinct': 'customers', 'key': 'city', 'query': {'loyalty': {'$ne': 'Bronze'}}} )
   -> {'values': ['Islamabad', 'Karachi', 'Lahore'], 'ok': 1} 

db.runCommand( {'create': 'wishlists'} )
   -> {'ok': 1} 

db.runCommand( {'listCollections': 1} )
   -> {'cursor': {'firstBatch': [{'name': 'customers'}, {'name': 'orders'}, {'name': 'products'}, {'name': 'wishlists'}]}, 'ok': 1} 

db.runCommand( {'createIndexes': 'products', 'indexes': [{'key': {'category': 1, 'price': -1}, 'name': 'cat_price'}]} )
   -> {'createdIndexes': ['cat_price'], 'ok': 1} 

db.runCommand( {'find': 'products', 'filter': {'stock': 0}, 'projection': {'name': 1}, 'limit': 1} )
   -> {'cursor': {'firstBatch': [{'name': 'USB-C Hub', '_id': 104}], 'ns': 'kgs.products'}, 'ok': 1} 

db.runCommand( {'drop': 'wishlists'} )
   -> {'ns': 'kgs.wishlists', 'ok': 1} 

db.runCommand( {'fsyncUnlockEverything': 1} )
